# <font color="#418FDE" size="6.5" uppercase>**Training Networks**</font>

>Last update: 20260709.
    
By the end of this Lecture, you will be able to:
- Explain gradient descent, stochastic gradient descent, and backpropagation at an intuitive level. 
- Implement a small neural network training loop from scratch using NumPy. 
- Evaluate neural network performance and identify signs of overfitting. 


## **1. Learning Mechanics**

### **1.1. Gradient Descent Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_07/Lecture_B/image_01_01.jpg?v=1783623942" width="250">



>* Networks adjust weights to reduce prediction error
>* Gradient descent takes small downhill loss steps

>* Gradients guide each parameter’s adjustment
>* Learning rate controls update step size

>* Training improves through repeated feedback cycles
>* Complex landscapes often yield good-enough solutions



In [ ]:
#@title Python Code - Gradient Descent Basics

# Gradient descent adjusts one parameter gradually.
# Loss measures prediction error before updating.
# Learning rate controls each downhill step.

import numpy as np
import matplotlib.pyplot as plt

# Create a tiny civil engineering dataset.
loads_kips = np.array([2, 4, 6, 8, 10], dtype=float)
settlement_inches = np.array([0.9, 1.7, 2.8, 3.6, 4.5], dtype=float)

# Validate matching one dimensional data shapes.
assert loads_kips.ndim == settlement_inches.ndim == 1
assert loads_kips.size == settlement_inches.size

# Scale inputs for stable gradient steps.
x_values = loads_kips / loads_kips.max()
y_values = settlement_inches / settlement_inches.max()

# Start with a poor slope estimate.
weight = 0.10
learning_rate = 0.40

# Store loss values for visualization.
loss_history = []
weight_history = []

# Run a short gradient descent loop.
for step_index in range(25):
    predictions = weight * x_values
    errors = predictions - y_values

    loss_value = np.mean(errors ** 2)
    gradient = 2 * np.mean(errors * x_values)

    weight = weight - learning_rate * gradient
    loss_history.append(loss_value)
    weight_history.append(weight)

# Convert learned scaled slope back to inches per kip.
learned_slope = weight * settlement_inches.max() / loads_kips.max()
final_loss = loss_history[-1]

# Print a compact training summary.
print(f"NumPy version: {np.__version__}")
print(f"Initial loss: {loss_history[0]:.4f}")
print(f"Final loss: {final_loss:.4f}")
print(f"Learned slope: {learned_slope:.3f} inches per kip")
print("Gradient descent repeatedly reduced prediction error.")

# Plot loss decreasing over training steps.
plt.figure(figsize=(7, 4))
plt.plot(loss_history, marker="o", linewidth=2)
plt.xlabel("Gradient descent step")
plt.ylabel("Mean squared error")
plt.title("Loss decreases as the parameter moves downhill")
plt.grid(True, alpha=0.3)
plt.show()



### **1.2. Stochastic Gradient Descent**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_07/Lecture_B/image_01_02.jpg?v=1783623939" width="250">



>* Updates weights using small random data samples
>* Noisy steps gradually reduce prediction loss

>* Small batches make learning faster
>* Frequent updates handle large datasets efficiently

>* Noise helps explore better solutions
>* Learning rate and batch size need balance



In [ ]:
#@title Python Code - Stochastic Gradient Descent

# This script compares batch and stochastic descent.
# Small concrete data keeps the mechanics visible.
# Civil engineering context uses beam deflection data.

import numpy as np
import matplotlib.pyplot as plt

# Set a seed for repeatable stochastic batches.
rng = np.random.default_rng(7)

# Create tiny synthetic beam load data.
loads_kN = np.linspace(10, 80, 40)
true_deflection_mm = 0.42 * loads_kN + 1.5
noise_mm = rng.normal(0, 1.8, loads_kN.size)

# Combine signal and measurement noise.
deflection_mm = true_deflection_mm + noise_mm

# Standardize inputs for stable gradient steps.
x_mean = loads_kN.mean()
x_std = loads_kN.std()
x = (loads_kN - x_mean) / x_std

# Validate matching one dimensional arrays.
assert x.shape == deflection_mm.shape
assert x.ndim == 1 and deflection_mm.ndim == 1

# Define mean squared error for one line.
def mse_loss(weight, bias, x_values, y_values):
    predictions = weight * x_values + bias
    errors = predictions - y_values
    return np.mean(errors ** 2)

# Define one gradient update using selected examples.
def gradient_step(weight, bias, x_batch, y_batch, rate):
    predictions = weight * x_batch + bias
    errors = predictions - y_batch
    grad_w = 2 * np.mean(errors * x_batch)

    grad_b = 2 * np.mean(errors)
    new_weight = weight - rate * grad_w
    new_bias = bias - rate * grad_b
    return new_weight, new_bias

# Train with full batch gradient descent.
full_w, full_b = 0.0, 0.0
full_losses = []
learning_rate = 0.08

for step in range(60):
    full_w, full_b = gradient_step(
        full_w, full_b, x, deflection_mm, learning_rate
    )
    full_losses.append(mse_loss(full_w, full_b, x, deflection_mm))

# Train with stochastic mini batches.
sgd_w, sgd_b = 0.0, 0.0
sgd_losses = []
batch_size = 5

for step in range(60):
    batch_ids = rng.choice(x.size, batch_size, replace=False)
    sgd_w, sgd_b = gradient_step(
        sgd_w, sgd_b, x[batch_ids], deflection_mm[batch_ids], learning_rate
    )
    sgd_losses.append(mse_loss(sgd_w, sgd_b, x, deflection_mm))

# Print a compact comparison of final losses.
print(f"NumPy version: {np.__version__}")
print(f"Full batch final loss: {full_losses[-1]:.2f}")
print(f"Mini batch SGD final loss: {sgd_losses[-1]:.2f}")
print("SGD is noisier, but usually moves downhill quickly.")

# Plot both loss paths for visual comparison.
plt.figure(figsize=(7, 4))
plt.plot(full_losses, label="Full batch gradient descent")
plt.plot(sgd_losses, label="Mini batch stochastic gradient descent")

# Add labels that connect curves to learning.
plt.xlabel("Update step")
plt.ylabel("Mean squared error")
plt.title("Many small noisy SGD steps can reduce loss")
plt.legend()

# Display the single teaching plot.
plt.tight_layout()
plt.show()



### **1.3. Backpropagation Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_07/Lecture_B/image_01_03.jpg?v=1783623944" width="250">



>* Measures prediction error after forward pass
>* Traces responsibility backward to adjust parameters

>* Measures each parameter’s effect on error
>* Traces influence backward through network layers

>* Backpropagation provides gradients for optimizer updates
>* Repeated cycles efficiently improve network predictions



In [ ]:
#@title Python Code - Backpropagation Basics

# Backpropagation traces error through network layers.
# This tiny example uses NumPy only.
# Civil data predicts safe beam behavior.

# NumPy and Matplotlib are already available.
import numpy as np
import matplotlib.pyplot as plt

# Set deterministic randomness for repeatable learning.
rng = np.random.default_rng(7)

# Create tiny civil engineering style features.
span_ft = np.array([8, 10, 12, 14, 16, 18], dtype=float)
load_kips = np.array([2, 3, 4, 5, 6, 7], dtype=float)

# Stack and scale inputs for stable gradients.
X_raw = np.column_stack([span_ft, load_kips])
X = (X_raw - X_raw.mean(axis=0)) / X_raw.std(axis=0)

# Labels mark acceptable deflection behavior.
y = np.array([[1], [1], [1], [0], [0], [0]], dtype=float)
assert X.shape == (6, 2) and y.shape == (6, 1)

# Initialize a small two layer network.
W1 = rng.normal(0, 0.4, size=(2, 3))
b1 = np.zeros((1, 3))
W2 = rng.normal(0, 0.4, size=(3, 1))
b2 = np.zeros((1, 1))

# Define sigmoid activation and its slope.
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Store losses for one compact plot.
losses = []
lr = 0.4

# Train silently using full batch gradient descent.
for epoch in range(120):
    z1 = X @ W1 + b1
    a1 = sigmoid(z1)
    z2 = a1 @ W2 + b2
    yhat = sigmoid(z2)

    # Measure prediction error with mean squared loss.
    loss = np.mean((yhat - y) ** 2)
    losses.append(loss)

    # Backpropagate output error first.
    dz2 = (2 / len(y)) * (yhat - y) * yhat * (1 - yhat)
    dW2 = a1.T @ dz2
    db2 = dz2.sum(axis=0, keepdims=True)

    # Backpropagate hidden layer responsibility next.
    dz1 = (dz2 @ W2.T) * a1 * (1 - a1)
    dW1 = X.T @ dz1
    db1 = dz1.sum(axis=0, keepdims=True)

    # Gradient descent updates move opposite gradients.
    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1

# Evaluate final predictions after learning.
final_probs = yhat.ravel()
final_classes = (final_probs >= 0.5).astype(int)
accuracy = np.mean(final_classes == y.ravel())

# Print only concise teaching results.
print(f"NumPy version: {np.__version__}")
print(f"Initial loss: {losses[0]:.3f}")
print(f"Final loss: {losses[-1]:.3f}")
print(f"Training accuracy: {accuracy:.0%}")
print("Backpropagation computed gradients before each update.")

# Plot loss curve to show learning progress.
plt.figure(figsize=(6, 4))
plt.plot(losses, color="navy", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Mean squared loss")
plt.title("Backpropagation reduces prediction error")
plt.grid(True, alpha=0.3)
plt.show()



## **2. Training Code**

### **2.1. NumPy Training Loop**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_07/Lecture_B/image_02_01.jpg?v=1783623923" width="250">



>* Training improves random weights through repeated feedback
>* NumPy reveals prediction, loss, gradients, updates

>* Forward pass predicts and stores layer outputs
>* Backpropagation updates parameters using learning rate

>* Use correct shapes and vectorized batches
>* Track loss to diagnose training progress



In [ ]:
#@title Python Code - NumPy Training Loop

# Build a tiny neural network manually.
# Use NumPy for every training step.
# Track loss to inspect learning behavior.

import numpy as np
import matplotlib.pyplot as plt

# Set a seed for repeatable results.
rng = np.random.default_rng(7)

# Create small civil engineering style data.
loads_kips = np.linspace(5, 80, 80).reshape(-1, 1)
spans_feet = np.linspace(10, 60, 80).reshape(-1, 1)

# Combine features into one design matrix.
X_raw = np.hstack([loads_kips, spans_feet])
y_raw = 0.03 * loads_kips + 0.015 * spans_feet

# Add mild noise to mimic measurements.
y_raw = y_raw + rng.normal(0, 0.08, y_raw.shape)

# Standardize inputs for stable gradient descent.
X_mean = X_raw.mean(axis=0, keepdims=True)
X_std = X_raw.std(axis=0, keepdims=True)
X = (X_raw - X_mean) / X_std

# Standardize targets for easier training.
y_mean = y_raw.mean(axis=0, keepdims=True)
y_std = y_raw.std(axis=0, keepdims=True)
y = (y_raw - y_mean) / y_std

# Split data into training and validation sets.
train_count = 60
X_train, X_val = X[:train_count], X[train_count:]
y_train, y_val = y[:train_count], y[train_count:]

# Validate shapes before matrix operations.
assert X_train.shape == (60, 2)
assert y_train.shape == (60, 1)

# Initialize a small one hidden layer network.
hidden_units = 6
W1 = rng.normal(0, 0.2, (2, hidden_units))
b1 = np.zeros((1, hidden_units))

# Initialize output layer parameters.
W2 = rng.normal(0, 0.2, (hidden_units, 1))
b2 = np.zeros((1, 1))

# Choose learning settings for quick training.
learning_rate = 0.03
epochs = 500
loss_history = []

# Train using full batch gradient descent.
for epoch in range(epochs):
    # Forward pass computes predictions.
    z1 = X_train @ W1 + b1
    a1 = np.tanh(z1)
    y_pred = a1 @ W2 + b2

    # Mean squared error measures prediction quality.
    error = y_pred - y_train
    loss = np.mean(error ** 2)
    loss_history.append(loss)

    # Backpropagation computes output gradients.
    n = X_train.shape[0]
    d_y_pred = 2 * error / n
    dW2 = a1.T @ d_y_pred
    db2 = d_y_pred.sum(axis=0, keepdims=True)

    # Backpropagation continues through hidden layer.
    d_a1 = d_y_pred @ W2.T
    d_z1 = d_a1 * (1 - a1 ** 2)
    dW1 = X_train.T @ d_z1
    db1 = d_z1.sum(axis=0, keepdims=True)

    # Gradient descent updates every parameter.
    W1 = W1 - learning_rate * dW1
    b1 = b1 - learning_rate * db1
    W2 = W2 - learning_rate * dW2
    b2 = b2 - learning_rate * db2

# Evaluate validation performance after training.
z1_val = X_val @ W1 + b1
a1_val = np.tanh(z1_val)
y_val_pred = a1_val @ W2 + b2

# Convert validation predictions back to original units.
y_val_real = y_val * y_std + y_mean
y_pred_real = y_val_pred * y_std + y_mean
val_rmse = np.sqrt(np.mean((y_pred_real - y_val_real) ** 2))

# Print concise training and evaluation results.
print(f"NumPy version: {np.__version__}")
print(f"Initial training loss: {loss_history[0]:.4f}")
print(f"Final training loss: {loss_history[-1]:.4f}")
print(f"Validation RMSE: {val_rmse:.3f} inches")
print("Large train improvement with poor validation suggests overfitting.")

# Plot the loss curve for visual inspection.
plt.figure(figsize=(6, 4))
plt.plot(loss_history, color="navy")
plt.xlabel("Epoch")
plt.ylabel("Training MSE")
plt.title("NumPy Neural Network Training Loop")
plt.grid(True, alpha=0.3)
plt.show()



### **2.2. Loss Computation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_07/Lecture_B/image_02_02.jpg?v=1783623926" width="250">



>* Loss measures prediction error for each task
>* Loss guides learning and gradient calculation

>* Match each prediction to its target
>* Average batch errors to track improvement

>* Keep loss calculations numerically stable
>* Use loss trends to diagnose learning



In [ ]:
#@title Python Code - Loss Computation

# This example computes losses for tiny batches.
# Civil engineering predictions use simple NumPy arrays.
# Stable loss values guide later backpropagation.

import numpy as np

# Set deterministic values for repeatable teaching results.
np.random.seed(7)

# Store regression targets in millimeters of settlement.
y_true_mm = np.array([[12.0], [18.0], [25.0], [31.0]])

# Store model predictions in matching target shape.
y_pred_mm = np.array([[10.5], [20.0], [22.0], [35.0]])

# Validate matching shapes before computing errors.
assert y_true_mm.shape == y_pred_mm.shape

# Compute mean squared error for regression.
squared_errors = (y_pred_mm - y_true_mm) ** 2
mse_loss = np.mean(squared_errors)

# Store raw class scores for three soil classes.
logits = np.array([[2.0, 0.5, -1.0], [0.1, 1.8, 0.2]])

# Store correct class indices for each sample.
y_class = np.array([0, 1])

# Validate classification batch and target lengths.
assert logits.shape[0] == y_class.shape[0]

# Shift logits to improve softmax numerical stability.
shifted_logits = logits - np.max(logits, axis=1, keepdims=True)

# Convert raw scores into class probabilities.
exp_scores = np.exp(shifted_logits)
probabilities = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

# Clip probabilities before taking logarithms safely.
safe_probs = np.clip(probabilities, 1e-12, 1.0)

# Select probabilities assigned to correct classes.
correct_probs = safe_probs[np.arange(y_class.size), y_class]

# Compute average cross entropy classification loss.
cross_entropy_loss = -np.mean(np.log(correct_probs))

# Print compact results for lecture discussion.
print(f"Regression MSE loss: {mse_loss:.2f} square millimeters")
print(f"Classification cross-entropy loss: {cross_entropy_loss:.3f}")
print(f"Correct-class probabilities: {np.round(correct_probs, 3)}")



### **2.3. Loss Computation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_07/Lecture_B/image_02_03.jpg?v=1783623928" width="250">



>* Loss measures prediction error for the task
>* Match shapes, labels, and loss function

>* Loss rewards correct confidence, penalizes mistakes
>* Average batch losses for stable training

>* Correct loss guides useful backpropagation gradients
>* Monitor loss to catch training problems



In [ ]:
#@title Python Code - Loss Computation

# Loss turns predictions into one training score.
# This example compares regression and classification losses.
# NumPy lets us compute losses from scratch.

import numpy as np

# Set print style for compact teaching output.
np.set_printoptions(precision=3, suppress=True)

# Create tiny civil engineering regression targets.
true_strength_mpa = np.array([[28.0], [35.0], [42.0]])
pred_strength_mpa = np.array([[30.0], [31.0], [45.0]])

# Validate matching regression array shapes.
assert true_strength_mpa.shape == pred_strength_mpa.shape

# Compute mean squared error for regression.
errors_mpa = pred_strength_mpa - true_strength_mpa
mse_loss = np.mean(errors_mpa ** 2)

# Create tiny classification probabilities for three classes.
class_names = np.array(["safe", "review", "unsafe"])
probabilities = np.array([[0.80, 0.15, 0.05], [0.10, 0.25, 0.65]])
true_classes = np.array([0, 2])

# Validate classification shapes before indexing.
assert probabilities.shape[0] == true_classes.size
assert probabilities.shape[1] == class_names.size

# Clip probabilities to avoid logarithm instability.
epsilon = 1e-12
safe_probabilities = np.clip(probabilities, epsilon, 1.0)

# Select probabilities assigned to correct classes.
row_indices = np.arange(true_classes.size)
correct_probabilities = safe_probabilities[row_indices, true_classes]

# Compute average cross entropy classification loss.
classification_losses = -np.log(correct_probabilities)
cross_entropy_loss = np.mean(classification_losses)

# Print concise results for interpretation.
print("NumPy version:", np.__version__)
print("Regression MSE loss:", round(float(mse_loss), 3))
print("Correct class probabilities:", correct_probabilities)
print("Classification cross entropy:", round(float(cross_entropy_loss), 3))
print("Lower loss means predictions better match targets.")



## **3. Network Evaluation**

### **3.1. Test Performance**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_07/Lecture_B/image_03_01.jpg?v=1783623930" width="250">



>* Test on data unseen during training
>* Measure generalization, not memorization

>* Choose metrics that match task risks
>* Judge errors by real-world reliability

>* Compare test results with training and validation
>* Use representative tests for realistic generalization



In [ ]:
#@title Python Code - Test Performance

# Test performance estimates future network behavior.
# Separate data reveals memorization and overfitting.
# Civil examples need realistic unseen cases.

import numpy as np
import matplotlib.pyplot as plt

# Set deterministic randomness for repeatable results.
rng = np.random.default_rng(7)

# Create synthetic bridge sensor measurements.
n_samples = 240
x = rng.uniform(-3.0, 3.0, n_samples)

# Define noisy deflection response in millimeters.
noise = rng.normal(0.0, 0.18, n_samples)
y = np.sin(x) + 0.25 * x + noise

# Shuffle before splitting into separate sets.
order = rng.permutation(n_samples)
x = x[order]
y = y[order]

# Reserve unseen data for final testing.
train_end = 150
test_start = 190
x_train, y_train = x[:train_end], y[:train_end]
x_test, y_test = x[test_start:], y[test_start:]

# Validate split sizes before fitting models.
assert x_train.size > 20 and x_test.size > 20
assert x_train.shape == y_train.shape

# Fit a simple model and an overflexible model.
simple_coef = np.polyfit(x_train, y_train, deg=2)
complex_coef = np.polyfit(x_train, y_train, deg=14)

# Predict training and test responses.
simple_train = np.polyval(simple_coef, x_train)
simple_test = np.polyval(simple_coef, x_test)
complex_train = np.polyval(complex_coef, x_train)
complex_test = np.polyval(complex_coef, x_test)

# Compute root mean squared error.
def rmse(actual_values, predicted_values):
    squared_errors = (actual_values - predicted_values) ** 2
    return float(np.sqrt(np.mean(squared_errors)))

# Summarize performance without printing arrays.
print("NumPy version:", np.__version__)
print("Metric: RMSE in millimeters, lower is better.")
print("Simple train RMSE:", round(rmse(y_train, simple_train), 3))
print("Simple test RMSE:", round(rmse(y_test, simple_test), 3))
print("Complex train RMSE:", round(rmse(y_train, complex_train), 3))
print("Complex test RMSE:", round(rmse(y_test, complex_test), 3))
print("Large train-test gap suggests overfitting.")

# Plot unseen test predictions for comparison.
grid = np.linspace(-3.0, 3.0, 200)
plt.figure(figsize=(7, 4))
plt.scatter(x_test, y_test, s=25, label="unseen test data")
plt.plot(grid, np.polyval(simple_coef, grid), label="simple model")
plt.plot(grid, np.polyval(complex_coef, grid), label="complex model")
plt.xlabel("Normalized load sensor reading")
plt.ylabel("Bridge deflection, millimeters")
plt.title("Test performance reveals generalization")
plt.legend()
plt.tight_layout()
plt.show()



### **3.2. Overfitting Signs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_07/Lecture_B/image_03_02.jpg?v=1783623936" width="250">



>* Overfitting memorizes training data quirks
>* Check performance on unseen data

>* Falling training loss, rising validation loss
>* Check performance across realistic unseen conditions

>* Overfit models react strongly to small changes
>* Check confidence, errors, and unseen data



In [ ]:
#@title Python Code - Overfitting Signs

# Demonstrate overfitting using synthetic civil data.
# Compare training and validation loss curves.
# Keep the NumPy network intentionally small.

import numpy as np
import matplotlib.pyplot as plt

# Print the numerical library version.
print("NumPy version:", np.__version__)

# Make results repeatable for learners.
rng = np.random.default_rng(7)

# Create a small noisy engineering dataset.
n_samples = 80
x = np.linspace(-3.0, 3.0, n_samples).reshape(-1, 1)

# Model a nonlinear settlement response.
noise = rng.normal(0.0, 0.18, size=(n_samples, 1))
y = np.sin(1.7 * x) + 0.25 * x + noise

# Split into training and validation sets.
train_count = 24
x_train, y_train = x[:train_count], y[:train_count]
x_val, y_val = x[train_count:], y[train_count:]

# Validate the split before training.
assert x_train.shape[0] == y_train.shape[0]
assert x_val.shape[0] == y_val.shape[0]

# Initialize a flexible one hidden layer network.
hidden = 40
W1 = rng.normal(0.0, 0.7, size=(1, hidden))
b1 = np.zeros((1, hidden))

# Initialize output layer parameters.
W2 = rng.normal(0.0, 0.7, size=(hidden, 1))
b2 = np.zeros((1, 1))

# Store losses for evaluation.
train_losses = []
val_losses = []

# Train long enough to reveal overfitting.
learning_rate = 0.015
epochs = 900

# Run silent gradient descent training.
for epoch in range(epochs):
    z1 = x_train @ W1 + b1
    a1 = np.tanh(z1)
    y_hat = a1 @ W2 + b2

    error = y_hat - y_train
    loss = float(np.mean(error ** 2))
    train_losses.append(loss)

    dz2 = (2.0 / train_count) * error
    dW2 = a1.T @ dz2
    db2 = np.sum(dz2, axis=0, keepdims=True)

    da1 = dz2 @ W2.T
    dz1 = da1 * (1.0 - a1 ** 2)
    dW1 = x_train.T @ dz1

    db1 = np.sum(dz1, axis=0, keepdims=True)
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2

    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    val_pred = np.tanh(x_val @ W1 + b1) @ W2 + b2

    val_loss = float(np.mean((val_pred - y_val) ** 2))
    val_losses.append(val_loss)

# Find the best validation epoch.
best_epoch = int(np.argmin(val_losses))
final_gap = val_losses[-1] - train_losses[-1]

# Print a compact overfitting summary.
print("Best validation epoch:", best_epoch)
print("Final training loss:", round(train_losses[-1], 4))
print("Final validation loss:", round(val_losses[-1], 4))
print("Validation minus training gap:", round(final_gap, 4))

# Interpret the warning sign plainly.
if best_epoch < epochs - 150 and final_gap > 0.05:
    print("Warning: validation worsened while training kept improving.")
else:
    print("No strong overfitting warning in this run.")

# Plot both losses to reveal divergence.
plt.figure(figsize=(7, 4))
plt.plot(train_losses, label="training loss")
plt.plot(val_losses, label="validation loss")

# Mark the best validation point.
plt.axvline(best_epoch, color="gray", linestyle="--", label="best validation")
plt.xlabel("Epoch")
plt.ylabel("Mean squared error")

# Add a clear teaching title.
plt.title("Overfitting sign: validation loss rises later")
plt.legend()
plt.tight_layout()
plt.show()



### **3.3. Overfitting Signs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/ML & AI for Civil Engineers/Module_07/Lecture_B/image_03_03.jpg?v=1783623934" width="250">



>* Model memorizes training quirks, not general patterns
>* Large train-test gaps signal poor generalization

>* Training improves while validation worsens
>* Validation curves reveal risky overfitting

>* Test errors on realistic input variations
>* Compare splits, trends, and error patterns



In [ ]:
#@title Python Code - Overfitting Signs

# This example shows overfitting signs.
# Training improves while validation worsens.
# Curves reveal weak generalization early.

# numpy and matplotlib are usually preinstalled.
import numpy as np
import matplotlib.pyplot as plt

# Print library version briefly.
print("NumPy version:", np.__version__)

# Make deterministic synthetic engineering data.
rng = np.random.default_rng(7)
x_all = np.linspace(-3.0, 3.0, 90).reshape(-1, 1)

# Create noisy settlement measurements in centimeters.
y_true = np.sin(2.2 * x_all) + 0.25 * x_all
noise = rng.normal(0.0, 0.18, size=y_true.shape)

# Combine signal and measurement noise.
y_all = y_true + noise
assert x_all.shape == y_all.shape

# Use few training points to encourage overfitting.
train_idx = np.arange(0, 90, 3)
valid_idx = np.setdiff1d(np.arange(90), train_idx)

# Split training and validation arrays.
x_train = x_all[train_idx]
y_train = y_all[train_idx]

# Keep validation data unseen during fitting.
x_valid = x_all[valid_idx]
y_valid = y_all[valid_idx]

# Build polynomial features for flexible fitting.
def poly_features(x_values, degree):
    powers = np.arange(degree + 1)
    return x_values ** powers

# Compute mean squared error safely.
def mse(y_actual, y_predicted):
    errors = y_actual - y_predicted
    return float(np.mean(errors ** 2))

# Compare simple and overly flexible models.
degrees = [3, 18]
results = []

# Fit each polynomial using least squares.
for degree in degrees:
    X_train = poly_features(x_train, degree)
    X_valid = poly_features(x_valid, degree)

    # Solve weights without machine learning libraries.
    weights = np.linalg.lstsq(X_train, y_train, rcond=None)[0]
    train_loss = mse(y_train, X_train @ weights)
    valid_loss = mse(y_valid, X_valid @ weights)

    # Store losses and predictions for plotting.
    X_all = poly_features(x_all, degree)
    predictions = X_all @ weights
    results.append((degree, train_loss, valid_loss, predictions))

# Print compact evaluation summary.
for degree, train_loss, valid_loss, _ in results:
    gap = valid_loss - train_loss
    print(f"Degree {degree}: train MSE={train_loss:.3f}, validation MSE={valid_loss:.3f}, gap={gap:.3f}")

# Identify the overfitting warning sign.
print("Warning sign: low training error with much higher validation error.")

# Plot data and fitted curves.
plt.figure(figsize=(7, 4))
plt.scatter(x_train, y_train, label="training points", s=35)
plt.scatter(x_valid, y_valid, label="validation points", s=18, alpha=0.45)

# Add both model curves.
for degree, _, _, predictions in results:
    plt.plot(x_all, predictions, label=f"degree {degree} fit")

# Label the evaluation plot.
plt.title("Overfitting sign: validation performance falls behind")
plt.xlabel("soil depth position, meters")
plt.ylabel("settlement reading, centimeters")

# Show legend and grid clearly.
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Training Networks**</font>


In this lecture, you learned to:
- Explain gradient descent, stochastic gradient descent, and backpropagation at an intuitive level. 
- Implement a small neural network training loop from scratch using NumPy. 
- Evaluate neural network performance and identify signs of overfitting. 

In the next Module (Module 8), we will go over 'None'